In [1]:
pip install langchain langchain-google-genai wikipedia duckduckgo-search beautifulsoup4 google-generativeai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from langchain.agents import initialize_agent, Tool
from langchain.agents.agent_types import AgentType
from langchain.tools import DuckDuckGoSearchRun
from langchain.utilities import WikipediaAPIWrapper
from langchain.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI
from datetime import datetime
import wikipedia
from bs4 import BeautifulSoup
import requests
import re
from pathlib import Path
from dotenv import load_dotenv

d:\Agentic Ai\LangChain_GenAi\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
from pathlib import Path
import os


load_dotenv()


GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)


In [4]:
# Testing the LLM
llm.invoke("Hello, how are you?")

AIMessage(content='I am doing well, thank you for asking!  How are you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--0d8e700a-7f84-4b79-94a7-f6d21ce73f14-0', usage_metadata={'input_tokens': 6, 'output_tokens': 17, 'total_tokens': 23, 'input_token_details': {'cache_read': 0}})

In [ ]:
from langchain.tools import tool

@tool
def extract_year_from_summary(text: str) -> str:
    """Extracts the founding year from a given text."""
    import re
    match = re.search(r"\b(19|20)\d{2}\b", text)
    if match:
        return f"Founding year is {match.group(0)}"
    return "Founding year not found"


@tool
def founding_year_tool(company: str) -> str:
    """Returns the number of years since the company was founded using Wikipedia info."""
    try:
        summary = wikipedia.summary(company)
        for line in summary.split('.'):
            if "founded" in line.lower():
                year = [int(word) for word in line.split() if word.isdigit() and 1800 < int(word) < datetime.now().year]
                if year:
                    founded_year = year[0]
                    years_ago = datetime.now().year - founded_year
                    return f"{company} was founded in {founded_year}, which is {years_ago} years ago."
        return "Could not find founding year."
    
    except Exception as e:
        return f"Error: {e}"

def news_summarizer_tool_factory(llm_instance):
    @tool
    def news_summarizer_tool(query: str) -> str:
        """Searches top 3 news articles and summarizes them using Gemini."""
        try:
            search = DuckDuckGoSearchRun()
            results = search.run(query + " site:news")
            urls = [line for line in results.split("\n") if "http" in line][:3]

            summaries = []
            for url in urls:
                try:
                    res = requests.get(url, timeout=5)
                    soup = BeautifulSoup(res.text, "html.parser")
                    text = ' '.join([p.text for p in soup.find_all('p')])

                    text = text[:4000] # Gemini has a token limit, so we truncate the text
                    

                    prompt = f"Summarize this news article in 3 bullet points:\n\n{text}"
                    summary = llm_instance.invoke(prompt).content
                    summaries.append(f" {url}\n{summary}")
                except:
                    summaries.append(f" Failed to summarize: {url}")
            return "\n\n".join(summaries)
        except Exception as e:
            return f"Error during news summarization: {e}"
    return news_summarizer_tool

# Built-in Tools
wiki_tool = Tool(
    name="Wikipedia Search",
    func=WikipediaAPIWrapper().run,
    description="Fetches a short summary from Wikipedia"
)

ddg_search = Tool(
    name="DuckDuckGo Web Search",
    func=DuckDuckGoSearchRun().run,
    description="Use this to search anything on the web"
)

news_tool = news_summarizer_tool_factory(llm)

tools = [
    wiki_tool,
    ddg_search,
    founding_year_tool,
    news_tool,
    extract_year_from_summary
]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

query = """
Get me a short summary about Nvidia, calculate how old the company is, and give a 3-point summary of its latest news.
"""

response = agent.invoke({"input": query})
print("\n\nFinal Output:\n", response["output"])